In [1]:
import tensorflow
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [5]:
import os
os.chdir(r'C:\Adithya\Necessary_Items\Python\GenAI - Course\annclassification')
os.getcwd()

'C:\\Adithya\\Necessary_Items\\Python\\GenAI - Course\\annclassification'

In [6]:
#load all the trained models
model=load_model('model.h5')

with open('label_encode_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('one_encoder_geo.pkl','rb') as file:
    lable_encoder_geo = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)


In [23]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}
input_data

{'CreditScore': 600,
 'Geography': 'France',
 'Gender': 'Male',
 'Age': 40,
 'Tenure': 3,
 'Balance': 60000,
 'NumOfProducts': 2,
 'HasCrCard': 1,
 'IsActiveMember': 1,
 'EstimatedSalary': 50000}

In [24]:
geo_encoder = lable_encoder_geo.transform([[input_data['Geography']]])
geo_encoder_df = pd.DataFrame(geo_encoder.toarray(),columns=lable_encoder_geo.get_feature_names_out(['Geography']))
geo_encoder_df

c:\Users\adith\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [25]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [26]:
#Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [27]:
#concatentaion with onehot encoded data
input_df = pd.concat([input_df.drop('Geography',axis=1),geo_encoder_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [28]:
#scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.47154541,  0.90911166,  0.09477172, -0.69844549, -0.29010416,
         0.80510537,  0.63367318,  0.95214374, -0.84805047,  0.98019606,
        -0.57581067, -0.56349184]])

In [29]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step


array([[0.02990098]], dtype=float32)

In [33]:
prediction_probability = prediction[0][0]
prediction_probability

0.029900985

In [35]:
if prediction_probability > 0.5:
    print('customer will churn')
else:
    print('customer will not churn')

customer will not churn


In [39]:
lable_encoder_geo.categories_[0]

array(['France', 'Germany', 'Spain'], dtype=object)